# Curve Sequence Forecasting
Gegeben K aufeinanderfolgende Kurven (z.B. Kurven 790–800), sagt das Modell
die nächsten M Kurven vorher (z.B. 801–900).

**Architektur:**
1. **CurveEncoder** : jede Kurve [500] → kompakter Vektor [d_model]
2. **Temporaler Transformer** : Sequenz von K Vektoren → M Vorhersage-Vektoren
3. **CurveDecoder** : jeden Vorhersage-Vektor [d_model] → Kurve [500]

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

## Schritt 1: Datensatz generieren

In [ ]:
np.random.seed(42)

n_curves = 1000
t        = np.linspace(0, 10, 500)
SEQ_LEN  = len(t)

# Degradationskurve D(i): von 1.5 bei i=0 auf 1.0 bei i=900, auf ~0 bei i=999
k_opt = 0.01109
A     = 0.5 / (np.exp(900 * k_opt) - 1)
C     = A + 1.5
i_all    = np.arange(n_curves)
D_values = np.clip(C - A * np.exp(k_opt * i_all), 0.01, None)

D_start     = D_values[0]
D_threshold = 1.0
soh_values  = np.clip((D_values - D_threshold) / (D_start - D_threshold), 0.0, 1.0).astype(np.float32)

T_values = np.random.uniform(0.5, 2.0, n_curves)

def pt2_step(t, K, T, D):
    w0 = 1.0 / T
    if D > 1.0:
        wd = w0 * np.sqrt(D**2 - 1)
        y  = K * (1 - np.exp(-D*w0*t) * (np.cosh(wd*t) + (D/np.sqrt(D**2-1))*np.sinh(wd*t)))
    elif D == 1.0:
        y  = K * (1 - (1 + w0*t) * np.exp(-w0*t))
    else:
        wd = w0 * np.sqrt(1 - D**2)
        y  = K * (1 - np.exp(-D*w0*t) * (np.cos(wd*t) + (D/np.sqrt(1-D**2))*np.sin(wd*t)))
    return y

dataset = np.zeros((n_curves, SEQ_LEN))
for i in range(n_curves):
    dataset[i] = pt2_step(t, 1.0, T_values[i], D_values[i]) + np.random.normal(0, 0.02, SEQ_LEN)

# Normierung pro Kurve
data_min     = dataset.min(axis=1, keepdims=True)
data_max     = dataset.max(axis=1, keepdims=True)
dataset_norm = (dataset - data_min) / (data_max - data_min + 1e-8)
dataset_norm = dataset_norm.astype(np.float32)

print(f"Dataset: {dataset_norm.shape}")
print(f"SoH range: {soh_values.min():.3f} – {soh_values.max():.3f}")

## Schritt 2: Sliding-Window Dataset

Aus 1000 Kurven werden Fenster der Grösse `K_in + M_out` gebildet.
- `K_in = 20` : Eingabe-Kurven
- `M_out = 20` : Vorherzusagende Kurven
- ergibt 960 gültige Fenster

In [ ]:
K_IN  = 20   # Anzahl bekannte Kurven als Input
M_OUT = 20   # Anzahl vorherzusagender Kurven

class CurveSeqDataset(Dataset):
    """
    Erstellt Sliding-Window Samples:
      x      : K_IN aufeinanderfolgende Kurven  [K_IN,  SEQ_LEN]
      y      : M_OUT nachfolgende Kurven         [M_OUT, SEQ_LEN]
    """
    def __init__(self, curves):
        self.curves = torch.tensor(curves, dtype=torch.float32)
        self.n      = len(curves) - K_IN - M_OUT + 1

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        x = self.curves[idx          : idx + K_IN]           # [K_IN,  SEQ_LEN]
        y = self.curves[idx + K_IN   : idx + K_IN + M_OUT]   # [M_OUT, SEQ_LEN]
        return x, y

# Train/Test-Split ueber Kurvenindizes (nicht shuffeln – Zeitreihe!)
split      = int(0.8 * n_curves)
train_data = dataset_norm[:split]
test_data  = dataset_norm[split - K_IN:]   # Ueberlappung noetig fuer erstes Testfenster

train_loader = DataLoader(CurveSeqDataset(train_data), batch_size=32, shuffle=True)
test_loader  = DataLoader(CurveSeqDataset(test_data),  batch_size=32, shuffle=False)

print(f"Train-Fenster: {len(CurveSeqDataset(train_data))}")
print(f"Test-Fenster:  {len(CurveSeqDataset(test_data))}")
x_ex, y_ex = next(iter(train_loader))
print(f"Batch x: {x_ex.shape}  →  y: {y_ex.shape}")

## Schritt 3: Modell

In [ ]:
class CurveEncoder(nn.Module):
    """Kodiert eine einzelne Kurve [SEQ_LEN] → Vektor [d_model]"""
    def __init__(self, seq_len=500, d_model=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(seq_len, 256), nn.ReLU(),
            nn.Linear(256, d_model), nn.ReLU(),
        )
    def forward(self, x):
        # x: [B, K, SEQ_LEN] → [B, K, d_model]
        return self.net(x)


class CurveDecoder(nn.Module):
    """Dekodiert Vektor [d_model] → Kurve [SEQ_LEN]"""
    def __init__(self, seq_len=500, d_model=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 256), nn.ReLU(),
            nn.Linear(256, seq_len),
        )
    def forward(self, x):
        # x: [B, M, d_model] → [B, M, SEQ_LEN]
        return self.net(x)


class CurveSequenceForecaster(nn.Module):
    """
    Encoder-Decoder Transformer ueber Kurvensequenzen.
    Input:  K_IN  Kurven  [B, K_IN,  SEQ_LEN]
    Output: M_OUT Kurven  [B, M_OUT, SEQ_LEN]
    """
    def __init__(self, seq_len=500, d_model=128, nhead=4, num_layers=2,
                 k_in=20, m_out=20, dropout=0.1):
        super().__init__()
        self.m_out   = m_out
        self.d_model = d_model

        self.encoder_embed = CurveEncoder(seq_len, d_model)
        self.decoder_embed = CurveEncoder(seq_len, d_model)  # fuer Teacher Forcing
        self.decoder_query = nn.Embedding(m_out, d_model)    # lernbare Query-Vektoren

        # Positional Encoding (einfach, ueber Kurvenindex)
        self.enc_pos = nn.Embedding(k_in,  d_model)
        self.dec_pos = nn.Embedding(m_out, d_model)

        transformer = nn.Transformer(
            d_model=d_model, nhead=nhead,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers,
            dim_feedforward=256, dropout=dropout,
            batch_first=True
        )
        self.transformer = transformer
        self.curve_decoder = CurveDecoder(seq_len, d_model)

    def forward(self, x):
        # x: [B, K_IN, SEQ_LEN]
        B = x.size(0)

        # Encoder: Kurven einbetten + Positions-Embedding addieren
        enc_emb = self.encoder_embed(x)                                    # [B, K_IN, d_model]
        pos_idx = torch.arange(x.size(1), device=x.device)
        enc_emb = enc_emb + self.enc_pos(pos_idx).unsqueeze(0)             # [B, K_IN, d_model]

        # Decoder: lernbare Query-Vektoren (was soll vorhergesagt werden?)
        dec_idx = torch.arange(self.m_out, device=x.device)
        dec_emb = self.decoder_query(dec_idx).unsqueeze(0).expand(B, -1, -1)  # [B, M_OUT, d_model]
        dec_emb = dec_emb + self.dec_pos(dec_idx).unsqueeze(0)

        # Transformer
        out = self.transformer(enc_emb, dec_emb)                           # [B, M_OUT, d_model]

        # Kurven dekodieren
        return self.curve_decoder(out)                                     # [B, M_OUT, SEQ_LEN]


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = CurveSequenceForecaster(seq_len=SEQ_LEN, k_in=K_IN, m_out=M_OUT).to(device)
print(f"Parameter: {sum(p.numel() for p in model.parameters()):,}")
print(f"Device: {device}")

# Schnelltest
with torch.no_grad():
    test_out = model(x_ex.to(device))
print(f"Output shape: {test_out.shape}  (erwartet: [{x_ex.size(0)}, {M_OUT}, {SEQ_LEN}])")

## Schritt 4: Training

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

def train_epoch(model, loader):
    model.train()
    total_loss = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        pred = model(x)                   # [B, M_OUT, SEQ_LEN]
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(x)
    return total_loss / len(loader.dataset)

@torch.no_grad()
def eval_epoch(model, loader):
    model.eval()
    total_loss = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        pred = model(x)
        total_loss += criterion(pred, y).item() * len(x)
    return total_loss / len(loader.dataset)

n_epochs   = 50
train_loss_hist = []
val_loss_hist   = []

for epoch in range(1, n_epochs + 1):
    tr = train_epoch(model, train_loader)
    va = eval_epoch(model, test_loader)
    scheduler.step()
    train_loss_hist.append(tr)
    val_loss_hist.append(va)
    if epoch % 10 == 0:
        print(f"Epoch {epoch:02d} | Train MSE: {tr:.5f} | Val MSE: {va:.5f}")

## Schritt 5: Modell speichern

In [ ]:
torch.save({
    'model_state_dict': model.state_dict(),
    'SEQ_LEN': SEQ_LEN,
    'K_IN':    K_IN,
    'M_OUT':   M_OUT,
}, 'curve_seq_model.pth')
np.save('dataset_norm.npy', dataset_norm)
print('Gespeichert.')

## Schritt 6: Auswertung – Loss

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(train_loss_hist, label='Train MSE')
plt.plot(val_loss_hist,   label='Val MSE')
plt.title('Training Loss – Kurvensequenz-Vorhersage')
plt.xlabel('Epoche'); plt.ylabel('MSE')
plt.legend(); plt.grid(True); plt.show()

## Schritt 7: Inference

**Plot 1:** Eine einzelne vorhergesagte Kurve mit Original-Overlay  
**Plot 2:** Mehrere ausgewählte vorhergesagte Kurven (z.B. +20, +40, +60, +80, +100) mit Original

In [ ]:
import torch, torch.nn as nn, numpy as np, matplotlib.pyplot as plt

# ── Modell-Klassen (Kopie fuer Standalone-Ausfuehrung) ───────────────────────
class CurveEncoder(nn.Module):
    def __init__(self, seq_len=500, d_model=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(seq_len, 256), nn.ReLU(), nn.Linear(256, d_model), nn.ReLU())
    def forward(self, x): return self.net(x)

class CurveDecoder(nn.Module):
    def __init__(self, seq_len=500, d_model=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, 256), nn.ReLU(), nn.Linear(256, seq_len))
    def forward(self, x): return self.net(x)

class CurveSequenceForecaster(nn.Module):
    def __init__(self, seq_len=500, d_model=128, nhead=4, num_layers=2, k_in=20, m_out=20, dropout=0.1):
        super().__init__()
        self.m_out   = m_out
        self.d_model = d_model
        self.encoder_embed = CurveEncoder(seq_len, d_model)
        self.decoder_query = nn.Embedding(m_out, d_model)
        self.enc_pos       = nn.Embedding(k_in,  d_model)
        self.dec_pos       = nn.Embedding(m_out, d_model)
        self.transformer   = nn.Transformer(d_model=d_model, nhead=nhead,
                                            num_encoder_layers=num_layers, num_decoder_layers=num_layers,
                                            dim_feedforward=256, dropout=dropout, batch_first=True)
        self.curve_decoder = CurveDecoder(seq_len, d_model)
        # decoder_embed nicht benoetigt in Inference
    def forward(self, x):
        B = x.size(0)
        enc_emb = self.encoder_embed(x) + self.enc_pos(torch.arange(x.size(1), device=x.device)).unsqueeze(0)
        dec_emb = self.decoder_query(torch.arange(self.m_out, device=x.device)).unsqueeze(0).expand(B,-1,-1)
        dec_emb = dec_emb + self.dec_pos(torch.arange(self.m_out, device=x.device)).unsqueeze(0)
        out     = self.transformer(enc_emb, dec_emb)
        return self.curve_decoder(out)

# ── Einstellungen ─────────────────────────────────────────────────────────────
start_idx = 800   # <- Startindex (letzte Input-Kurve)

# ── Laden ─────────────────────────────────────────────────────────────────────
ckpt     = torch.load('curve_seq_model.pth', weights_only=True)
SEQ_LEN  = ckpt['SEQ_LEN']
K_IN     = ckpt['K_IN']
M_OUT    = ckpt['M_OUT']
device   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model    = CurveSequenceForecaster(seq_len=SEQ_LEN, k_in=K_IN, m_out=M_OUT).to(device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
dataset_norm = np.load('dataset_norm.npy')

# ── Vorhersage: naechste M_OUT Kurven autorekursiv bis +100 ──────────────────
# Wir wiederholen die Vorhersage, bis wir 100 Kurven haben
all_predicted = []          # gesammelte vorhergesagte Kurven
input_window  = list(dataset_norm[start_idx - K_IN + 1 : start_idx + 1])  # letzte K_IN echte Kurven

steps_needed = 100
while len(all_predicted) < steps_needed:
    x_in  = torch.tensor(np.array(input_window[-K_IN:]), dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        pred = model(x_in).squeeze(0).cpu().numpy()   # [M_OUT, SEQ_LEN]
    all_predicted.extend(pred)
    input_window.extend(pred)   # vorhergesagte Kurven als neuer Input

all_predicted = np.array(all_predicted[:steps_needed])   # [100, SEQ_LEN]

# ── Plot 1: Einzelkurven-Vergleich ────────────────────────────────────────────
highlight = [20, 40, 60, 80, 100]   # relative Offsets ab start_idx
fig, axes = plt.subplots(len(highlight), 1, figsize=(11, 3 * len(highlight)), sharex=True)

for ax, offset in zip(axes, highlight):
    true_idx = start_idx + offset
    if true_idx < len(dataset_norm):
        ax.plot(dataset_norm[true_idx], color='steelblue', alpha=0.8, label=f'Original (Kurve {true_idx})')
    ax.plot(all_predicted[offset - 1], color='tomato', linestyle='--', label=f'Vorhersage (+{offset})')
    ax.set_ylabel('Amplitude'); ax.legend(fontsize=8); ax.grid(True)

axes[-1].set_xlabel('Zeitschritte')
fig.suptitle(f'Kurvenvorhersage ab Index {start_idx}: Original vs. Vorhersage', fontsize=12)
plt.tight_layout(); plt.show()

# ── Plot 2: Heatmap – alle 100 vorhergesagten Kurven ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

true_range = dataset_norm[start_idx + 1 : start_idx + 101]
im0 = axes[0].imshow(true_range, aspect='auto', origin='upper', cmap='viridis')
axes[0].set_title(f'Original: Kurven {start_idx+1}–{start_idx+100}')
axes[0].set_xlabel('Zeitschritte'); axes[0].set_ylabel('Kurvenindex (relativ)')
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(all_predicted, aspect='auto', origin='upper', cmap='viridis')
axes[1].set_title(f'Vorhersage: Kurven {start_idx+1}–{start_idx+100}')
axes[1].set_xlabel('Zeitschritte'); axes[1].set_ylabel('Kurvenindex (relativ)')
plt.colorbar(im1, ax=axes[1])

plt.tight_layout(); plt.show()

# ── MSE pro vorhergesagter Kurve ──────────────────────────────────────────────
if start_idx + 100 < len(dataset_norm):
    mse_per_step = np.mean((all_predicted - true_range) ** 2, axis=1)
    plt.figure(figsize=(8, 3))
    plt.plot(range(1, 101), mse_per_step)
    plt.title('MSE pro vorhergesagter Kurve (steigt mit Vorhersagehorizont)')
    plt.xlabel('Schritte ab Kurve 800'); plt.ylabel('MSE'); plt.grid(True); plt.show()
    print(f"Mittlerer MSE ueber 100 Kurven: {mse_per_step.mean():.5f}")